# Bronze → Silver: Clean Raw Signals

This notebook reads raw signal data from the **bronze** layer (`bronze.raw_lifetime`, `bronze.raw_piece_info`), applies all cleaning rules, and writes validated pieces to the **silver** layer (`silver.clean_pieces`).

**Incremental**: only processes rows with timestamps newer than the latest entry in silver.

### All cleaning happens here

Silver must contain **only valid pieces**. The cleaning rules are:

1. Drop the upsetting signal (incorrectly calculated at the PLC)
2. Remove zero values (tracking failures)
3. Deduplicate timestamps per signal
4. Pivot lifetime signals into columns and join with piece identification
5. Drop rows missing piece_id or die_matrix
6. Remove extreme outliers (Q3 + 3×IQR per signal per die matrix)
7. Validate monotonic cumulative order: 2nd strike < 3rd strike < 4th strike < auxiliary press < bath
8. Compute OEE cycle time (rolling average of last 5 inter-piece intervals) and filter to 11–16s

See [03_CleaningUpData.md](../docs/03_CleaningUpData.md) for the full rationale behind each rule.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# -----------------------------
# Load environment variables
# -----------------------------
env_path = Path("../infra/.env")
if not env_path.exists():
    raise FileNotFoundError(f"Could not find env file: {env_path.resolve()}")

env = {}
for line in env_path.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    k, v = line.split("=", 1)
    env[k.strip()] = v.strip()

required = ["POSTGRES_PORT", "POSTGRES_DB", "POSTGRES_USER", "POSTGRES_PASSWORD"]
missing = [k for k in required if k not in env]
if missing:
    raise ValueError(f"Missing keys in infra/.env: {missing}")

# -----------------------------
# Database connection
# -----------------------------
engine = create_engine(
    f"postgresql+psycopg2://{env['POSTGRES_USER']}:{env['POSTGRES_PASSWORD']}"
    f"@localhost:{env['POSTGRES_PORT']}/{env['POSTGRES_DB']}"
)

with engine.connect() as conn:
    current_db = conn.execute(text("SELECT current_database()")).scalar()
    print("Connected to PostgreSQL:", current_db)

# -----------------------------
# Signal definitions
# -----------------------------
UPSETTING_SIGNAL = "forging_line.main_press.maintenance.forging_line_upsetting_lifetime_piecedata"

LIFETIME_SIGNAL_MAP = {
    "forging_line.main_press.maintenance.forging_line_lifetime_first_piecedata": "lifetime_2nd_strike_s",
    "forging_line.main_press.maintenance.forging_line_lifetime_second_piecedata": "lifetime_3rd_strike_s",
    "forging_line.main_press.maintenance.forging_line_lifetime_drill_piecedata": "lifetime_4th_strike_s",
    "forging_line.auxiliary_press.maintenance.forging_line_lifetime_auxiliary_press_piecedata": "lifetime_auxiliary_press_s",
    "forging_line.bath.maintenance.forging_line_lifetime_bath_piecedata": "lifetime_bath_s",
    "forging_line.general.maintenance.forging_line_lifetime_piecedata": "lifetime_general_s",
}

PIECE_INFO_SIGNAL_MAP = {
    "forging_line.general.general.forging_line_piece_id_piecedata": "piece_id",
    "forging_line.general.general.forging_line_die_matrix_piecedata": "die_matrix",
}

# -----------------------------
# Cleaning report accumulator
# -----------------------------
cleaning_report = {
    "silver_watermark": None,
    "raw_lifetime_rows_extracted": 0,
    "raw_piece_info_rows_extracted": 0,
    "upsetting_rows_removed": 0,
    "zero_rows_removed": 0,
    "near_duplicate_rows_removed": 0,
}


Connected to PostgreSQL: vaultech


## Step 1: Determine incremental boundary

Find the latest timestamp already processed in silver. Only bronze rows after this point will be processed.

In [2]:
silver_exists_sql = """
SELECT EXISTS (
    SELECT 1
    FROM information_schema.tables
    WHERE table_schema = 'silver'
      AND table_name = 'clean_pieces'
) AS table_exists
"""

with engine.connect() as conn:
    silver_exists = conn.execute(text(silver_exists_sql)).scalar()

if silver_exists:
    watermark_df = pd.read_sql("""
        SELECT MAX(timestamp) AS max_timestamp
        FROM silver.clean_pieces
    """, engine)
    silver_watermark = watermark_df.loc[0, "max_timestamp"]
else:
    silver_watermark = None

silver_watermark = pd.to_datetime(silver_watermark, utc=True, errors="coerce")
cleaning_report["silver_watermark"] = silver_watermark

print("Silver table exists:", silver_exists)
print("Incremental watermark:", silver_watermark)


Silver table exists: True
Incremental watermark: None


## Step 2: Extract and filter raw signals

Read from `bronze.raw_lifetime` excluding:
- The **upsetting signal** — incorrectly calculated at the PLC, values are meaningless (range 0–6.7s with 22.8% zeros)
- **Zero values** — tracking failures where the PLC did not detect the piece at a stage

In [3]:
if pd.isna(silver_watermark):
    raw_lifetime = pd.read_sql("""
        SELECT timestamp, signal, value
        FROM bronze.raw_lifetime
        ORDER BY timestamp, signal
    """, engine)
else:
    raw_lifetime = pd.read_sql("""
        SELECT timestamp, signal, value
        FROM bronze.raw_lifetime
        WHERE timestamp > %(watermark)s
        ORDER BY timestamp, signal
    """, engine, params={"watermark": silver_watermark.to_pydatetime()})

raw_lifetime["timestamp"] = pd.to_datetime(raw_lifetime["timestamp"], utc=True, errors="coerce")
raw_lifetime["value"] = pd.to_numeric(raw_lifetime["value"], errors="coerce")

cleaning_report["raw_lifetime_rows_extracted"] = len(raw_lifetime)

print("Extracted raw_lifetime rows:", len(raw_lifetime))
display(raw_lifetime.head(10))


Extracted raw_lifetime rows: 1233541


,timestamp,signal,value
0,2025-11-06 15:25:02.416000+00:00,forging_line.auxiliary_press.maintenance.forgi...,68.699997
1,2025-11-06 15:25:02.416000+00:00,forging_line.bath.maintenance.forging_line_lif...,70.300003
2,2025-11-06 15:25:02.416000+00:00,forging_line.general.maintenance.forging_line_...,70.300003
3,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,52.099998
4,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,32.000000
5,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,38.700001
6,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,0.200000
7,2025-11-06 15:25:16.426000+00:00,forging_line.auxiliary_press.maintenance.forgi...,54.599998
8,2025-11-06 15:25:16.426000+00:00,forging_line.bath.maintenance.forging_line_lif...,56.200001
9,2025-11-06 15:25:16.426000+00:00,forging_line.general.maintenance.forging_line_...,56.200001


In [4]:
# --- Count rows to be removed from raw_lifetime ---
upsetting_mask = raw_lifetime["signal"].eq(UPSETTING_SIGNAL)
zero_mask = raw_lifetime["value"].eq(0)

cleaning_report["upsetting_rows_removed"] = int(upsetting_mask.sum())
cleaning_report["zero_rows_removed"] = int((~upsetting_mask & zero_mask).sum())

# --- Apply filters ---
raw_lifetime = raw_lifetime.loc[~upsetting_mask].copy()
raw_lifetime = raw_lifetime.loc[raw_lifetime["value"] != 0].copy()

# Add readable signal names
raw_lifetime["signal_name"] = raw_lifetime["signal"].map(LIFETIME_SIGNAL_MAP).fillna(raw_lifetime["signal"])

# --- Extract raw_piece_info incrementally using same watermark ---
if pd.isna(silver_watermark):
    raw_piece_info = pd.read_sql("""
        SELECT timestamp, signal, value
        FROM bronze.raw_piece_info
        ORDER BY timestamp, signal
    """, engine)
else:
    raw_piece_info = pd.read_sql("""
        SELECT timestamp, signal, value
        FROM bronze.raw_piece_info
        WHERE timestamp > %(watermark)s
        ORDER BY timestamp, signal
    """, engine, params={"watermark": silver_watermark.to_pydatetime()})

raw_piece_info["timestamp"] = pd.to_datetime(raw_piece_info["timestamp"], utc=True, errors="coerce")
raw_piece_info["signal_name"] = raw_piece_info["signal"].map(PIECE_INFO_SIGNAL_MAP).fillna(raw_piece_info["signal"])

cleaning_report["raw_piece_info_rows_extracted"] = len(raw_piece_info)

print("After filtering raw_lifetime:")
print("  remaining rows:", len(raw_lifetime))
print("  upsetting removed:", cleaning_report["upsetting_rows_removed"])
print("  zero-value rows removed:", cleaning_report["zero_rows_removed"])
print()
print("Extracted raw_piece_info rows:", len(raw_piece_info))

display(raw_lifetime.head(10))
display(raw_piece_info.head(10))

After filtering raw_lifetime:
  remaining rows: 1041278
  upsetting removed: 179628
  zero-value rows removed: 12635

Extracted raw_piece_info rows: 359534


,timestamp,signal,value,signal_name
0,2025-11-06 15:25:02.416000+00:00,forging_line.auxiliary_press.maintenance.forgi...,68.699997,lifetime_auxiliary_press_s
1,2025-11-06 15:25:02.416000+00:00,forging_line.bath.maintenance.forging_line_lif...,70.300003,lifetime_bath_s
2,2025-11-06 15:25:02.416000+00:00,forging_line.general.maintenance.forging_line_...,70.300003,lifetime_general_s
3,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,52.099998,lifetime_4th_strike_s
4,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,32.000000,lifetime_2nd_strike_s
5,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,38.700001,lifetime_3rd_strike_s
7,2025-11-06 15:25:16.426000+00:00,forging_line.auxiliary_press.maintenance.forgi...,54.599998,lifetime_auxiliary_press_s
8,2025-11-06 15:25:16.426000+00:00,forging_line.bath.maintenance.forging_line_lif...,56.200001,lifetime_bath_s
9,2025-11-06 15:25:16.426000+00:00,forging_line.general.maintenance.forging_line_...,56.200001,lifetime_general_s
10,2025-11-06 15:25:16.426000+00:00,forging_line.main_press.maintenance.forging_li...,38.000000,lifetime_4th_strike_s


,timestamp,signal,value,signal_name
0,2025-11-06 15:25:02.416000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
1,2025-11-06 15:25:02.416000+00:00,forging_line.general.general.forging_line_piec...,251106001720,piece_id
2,2025-11-06 15:25:16.426000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
3,2025-11-06 15:25:16.426000+00:00,forging_line.general.general.forging_line_piec...,251106001721,piece_id
4,2025-11-06 15:25:29.134000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
5,2025-11-06 15:25:29.134000+00:00,forging_line.general.general.forging_line_piec...,251106001722,piece_id
6,2025-11-06 15:25:43.743000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
7,2025-11-06 15:25:43.743000+00:00,forging_line.general.general.forging_line_piec...,251106001723,piece_id
8,2025-11-06 15:25:56.462000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
9,2025-11-06 15:25:56.462000+00:00,forging_line.general.general.forging_line_piec...,251106001724,piece_id


## Step 3: Deduplicate timestamps

The PLC occasionally double-registers the same piece at the same timestamp. Keep only the last reading per signal.

In [5]:
lifetime_before = len(raw_lifetime)
piece_info_before = len(raw_piece_info)

# Keep only the last reading per (signal, timestamp)
raw_lifetime = (
    raw_lifetime
    .sort_values(["signal", "timestamp"])
    .drop_duplicates(subset=["signal", "timestamp"], keep="last")
    .reset_index(drop=True)
)

raw_piece_info = (
    raw_piece_info
    .sort_values(["signal", "timestamp"])
    .drop_duplicates(subset=["signal", "timestamp"], keep="last")
    .reset_index(drop=True)
)

removed_lifetime_dupes = lifetime_before - len(raw_lifetime)
removed_piece_info_dupes = piece_info_before - len(raw_piece_info)

cleaning_report["near_duplicate_rows_removed"] = int(
    removed_lifetime_dupes + removed_piece_info_dupes
)

print("Deduplication summary")
print("  raw_lifetime duplicates removed:", removed_lifetime_dupes)
print("  raw_piece_info duplicates removed:", removed_piece_info_dupes)
print("  total duplicates removed:", cleaning_report["near_duplicate_rows_removed"])

display(raw_lifetime.head(10))
display(raw_piece_info.head(10))


Deduplication summary
  raw_lifetime duplicates removed: 6
  raw_piece_info duplicates removed: 2
  total duplicates removed: 8


,timestamp,signal,value,signal_name
0,2025-11-06 15:25:02.416000+00:00,forging_line.auxiliary_press.maintenance.forgi...,68.699997,lifetime_auxiliary_press_s
1,2025-11-06 15:25:16.426000+00:00,forging_line.auxiliary_press.maintenance.forgi...,54.599998,lifetime_auxiliary_press_s
2,2025-11-06 15:25:29.134000+00:00,forging_line.auxiliary_press.maintenance.forgi...,54.799999,lifetime_auxiliary_press_s
3,2025-11-06 15:25:43.743000+00:00,forging_line.auxiliary_press.maintenance.forgi...,55.299999,lifetime_auxiliary_press_s
4,2025-11-06 15:25:56.462000+00:00,forging_line.auxiliary_press.maintenance.forgi...,55.500000,lifetime_auxiliary_press_s
5,2025-11-06 15:26:10.462000+00:00,forging_line.auxiliary_press.maintenance.forgi...,55.500000,lifetime_auxiliary_press_s
6,2025-11-06 15:26:23.771000+00:00,forging_line.auxiliary_press.maintenance.forgi...,53.599998,lifetime_auxiliary_press_s
7,2025-11-06 15:26:36.382000+00:00,forging_line.auxiliary_press.maintenance.forgi...,54.700001,lifetime_auxiliary_press_s
8,2025-11-06 15:26:50.095000+00:00,forging_line.auxiliary_press.maintenance.forgi...,54.700001,lifetime_auxiliary_press_s
9,2025-11-06 15:27:17.711000+00:00,forging_line.auxiliary_press.maintenance.forgi...,68.099998,lifetime_auxiliary_press_s


,timestamp,signal,value,signal_name
0,2025-11-06 15:25:02.416000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
1,2025-11-06 15:25:16.426000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
2,2025-11-06 15:25:29.134000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
3,2025-11-06 15:25:43.743000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
4,2025-11-06 15:25:56.462000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
5,2025-11-06 15:26:10.462000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
6,2025-11-06 15:26:23.771000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
7,2025-11-06 15:26:36.382000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
8,2025-11-06 15:26:50.095000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix
9,2025-11-06 15:27:17.711000+00:00,forging_line.general.general.forging_line_die_...,5052.0,die_matrix


## Step 4: Pivot and join

Transform the tall signal/value format into one row per piece with all cumulative times as columns. Join lifetime data with piece identification on timestamp.

In [6]:
# Pivot lifetime signals into wide format
lifetime_wide = (
    raw_lifetime.pivot_table(
        index="timestamp",
        columns="signal_name",
        values="value",
        aggfunc="last"
    )
    .reset_index()
)

# Pivot piece identification signals into wide format
piece_info_wide = (
    raw_piece_info.pivot_table(
        index="timestamp",
        columns="signal_name",
        values="value",
        aggfunc="last"
    )
    .reset_index()
)

# Flatten column index if needed
lifetime_wide.columns.name = None
piece_info_wide.columns.name = None

# Join on timestamp
pieces = lifetime_wide.merge(
    piece_info_wide,
    on="timestamp",
    how="left"
)

print("lifetime_wide shape:", lifetime_wide.shape)
print("piece_info_wide shape:", piece_info_wide.shape)
print("joined pieces shape:", pieces.shape)

display(pieces.head(10))
print("columns:", pieces.columns.tolist())


lifetime_wide shape: (183452, 7)
piece_info_wide shape: (179766, 3)
joined pieces shape: (183452, 9)


,timestamp,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,die_matrix,piece_id
0,2025-11-06 15:25:02.416000+00:00,32.000000,38.700001,52.099998,68.699997,70.300003,70.300003,5052.0,251106001720
1,2025-11-06 15:25:16.426000+00:00,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,5052.0,251106001721
2,2025-11-06 15:25:29.134000+00:00,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,5052.0,251106001722
3,2025-11-06 15:25:43.743000+00:00,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,5052.0,251106001723
4,2025-11-06 15:25:56.462000+00:00,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,5052.0,251106001724
5,2025-11-06 15:26:10.462000+00:00,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,5052.0,251106001725
6,2025-11-06 15:26:23.771000+00:00,16.700001,23.400000,36.799999,53.599998,55.200001,55.200001,5052.0,251106001726
7,2025-11-06 15:26:36.382000+00:00,18.299999,24.900000,38.200001,54.700001,56.400002,56.400002,5052.0,251106001727
8,2025-11-06 15:26:50.095000+00:00,17.900000,24.500000,37.700001,54.700001,56.299999,56.299999,5052.0,251106001728
9,2025-11-06 15:27:17.711000+00:00,16.799999,23.500000,51.099998,68.099998,69.699997,69.699997,5052.0,251106001729


columns: ['timestamp', 'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s', 'lifetime_4th_strike_s', 'lifetime_auxiliary_press_s', 'lifetime_bath_s', 'lifetime_general_s', 'die_matrix', 'piece_id']


## Step 5: Normalize and drop missing identification

Map column names to the silver schema, cast die_matrix to integer, and remove pieces missing piece_id or die_matrix.

In [7]:
# Keep only expected silver columns if present
expected_cols = [
    "timestamp",
    "piece_id",
    "die_matrix",
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
    "lifetime_general_s",
]

existing_cols = [c for c in expected_cols if c in pieces.columns]
pieces = pieces[existing_cols].copy()

# Normalize types
pieces["timestamp"] = pd.to_datetime(pieces["timestamp"], utc=True, errors="coerce")

if "piece_id" in pieces.columns:
    pieces["piece_id"] = pieces["piece_id"].astype("string").str.strip()

if "die_matrix" in pieces.columns:
    pieces["die_matrix"] = pd.to_numeric(pieces["die_matrix"], errors="coerce").astype("Int64")

# Count missing identification before dropping
missing_piece_id = pieces["piece_id"].isna() | (pieces["piece_id"].str.len() == 0)
missing_die_matrix = pieces["die_matrix"].isna()

cleaning_report["rows_missing_piece_id"] = int(missing_piece_id.sum())
cleaning_report["rows_missing_die_matrix"] = int(missing_die_matrix.sum())
cleaning_report["rows_missing_identification"] = int((missing_piece_id | missing_die_matrix).sum())

# Drop rows missing required identification
pieces = pieces.loc[~(missing_piece_id | missing_die_matrix)].copy()

print("Rows removed due to missing piece_id:", cleaning_report["rows_missing_piece_id"])
print("Rows removed due to missing die_matrix:", cleaning_report["rows_missing_die_matrix"])
print("Rows removed due to missing identification (union):", cleaning_report["rows_missing_identification"])
print("Remaining pieces after identification filter:", len(pieces))

display(pieces.head(10))
print(pieces.dtypes)


Rows removed due to missing piece_id: 5144
Rows removed due to missing die_matrix: 5144
Rows removed due to missing identification (union): 5144
Remaining pieces after identification filter: 178308


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s
0,2025-11-06 15:25:02.416000+00:00,251106001720,5052,32.000000,38.700001,52.099998,68.699997,70.300003,70.300003
1,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001
2,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002
3,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002
4,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998
5,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001
6,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,23.400000,36.799999,53.599998,55.200001,55.200001
7,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,24.900000,38.200001,54.700001,56.400002,56.400002
8,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,24.500000,37.700001,54.700001,56.299999,56.299999
9,2025-11-06 15:27:17.711000+00:00,251106001729,5052,16.799999,23.500000,51.099998,68.099998,69.699997,69.699997


timestamp                     datetime64[ns, UTC]
piece_id                           string[python]
die_matrix                                  Int64
lifetime_2nd_strike_s                     float64
lifetime_3rd_strike_s                     float64
lifetime_4th_strike_s                     float64
lifetime_auxiliary_press_s                float64
lifetime_bath_s                           float64
lifetime_general_s                        float64
dtype: object


## Step 6: Remove extreme outliers per die matrix

Pieces with cumulative times far outside the normal range are not slow pieces — they are **tracking failures**: stuck pieces, unclosed cycles, or machine stops that inflated the timer.

For each lifetime column, compute Q1, Q3, and IQR **per die matrix**, then remove rows where any value falls outside `[Q1 - 3×IQR, Q3 + 3×IQR]`.

In [8]:
lifetime_cols = [
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
    "lifetime_general_s",
]

# Only keep columns that actually exist
lifetime_cols = [c for c in lifetime_cols if c in pieces.columns]

pieces_before_outlier = len(pieces)

outlier_masks = []
outlier_thresholds = []

for matrix, group in pieces.groupby("die_matrix"):
    group_mask = pd.Series(False, index=group.index)

    for col in lifetime_cols:
        s = group[col].dropna()

        # Skip if column has too few non-null values to define IQR sensibly
        if len(s) < 4:
            continue

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 3 * iqr
        upper = q3 + 3 * iqr

        # Store thresholds for reporting
        outlier_thresholds.append({
            "die_matrix": int(matrix),
            "signal": col,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower,
            "upper_bound": upper,
        })

        col_mask = (group[col] < lower) | (group[col] > upper)
        col_mask = col_mask.fillna(False)

        group_mask = group_mask | col_mask

    outlier_masks.append(group_mask)

# Combine all per-matrix masks
if outlier_masks:
    all_outlier_mask = pd.concat(outlier_masks).sort_index()
else:
    all_outlier_mask = pd.Series(False, index=pieces.index)

cleaning_report["rows_removed_outliers"] = int(all_outlier_mask.sum())

pieces = pieces.loc[~all_outlier_mask].copy()

outlier_thresholds_df = pd.DataFrame(outlier_thresholds)

print("Rows removed as outliers:", cleaning_report["rows_removed_outliers"])
print("Remaining pieces after outlier removal:", len(pieces))

display(outlier_thresholds_df.head(20))
display(pieces.head(10))


Rows removed as outliers: 9154
Remaining pieces after outlier removal: 169154


,die_matrix,signal,q1,q3,iqr,lower_bound,upper_bound
0,4974,lifetime_2nd_strike_s,16.600000,18.500000,1.900000,10.900002,24.199999
1,4974,lifetime_3rd_strike_s,23.200001,25.100000,1.900000,17.500002,30.799999
2,4974,lifetime_4th_strike_s,36.500000,38.299999,1.799999,31.100002,43.699997
3,4974,lifetime_auxiliary_press_s,53.500000,55.599998,2.099998,47.200005,61.899994
4,4974,lifetime_bath_s,55.299999,57.299999,2.000000,49.299999,63.299999
5,4974,lifetime_general_s,55.299999,57.299999,2.000000,49.299999,63.299999
6,5052,lifetime_2nd_strike_s,17.400000,19.700001,2.300001,10.499996,26.600004
7,5052,lifetime_3rd_strike_s,24.299999,26.700001,2.400002,17.099995,33.900005
8,5052,lifetime_4th_strike_s,38.000000,40.900002,2.900002,29.299995,49.600006
9,5052,lifetime_auxiliary_press_s,55.099998,58.500000,3.400002,44.899994,68.700005


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s
1,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001
2,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002
3,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002
4,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998
5,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001
6,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,23.400000,36.799999,53.599998,55.200001,55.200001
7,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,24.900000,38.200001,54.700001,56.400002,56.400002
8,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,24.500000,37.700001,54.700001,56.299999,56.299999
12,2025-11-06 15:27:57.427000+00:00,251106001733,5052,16.700001,23.299999,36.599998,53.299999,54.900002,54.900002
16,2025-11-06 15:29:04.470000+00:00,251106001738,5052,16.700001,23.600000,36.799999,54.099998,55.799999,55.799999


## Step 7: Validate monotonic cumulative order

Each piece must have increasing cumulative times along the physical process:

**2nd strike < 3rd strike < 4th strike < auxiliary press < bath**

A violation means the PLC misattributed a reading or a tracking cycle overlapped. These are not valid pieces.

In [9]:
monotonic_cols = [
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
]

# Keep only columns that actually exist
monotonic_cols = [c for c in monotonic_cols if c in pieces.columns]

pieces_before_monotonic = len(pieces)

# A piece is monotonic only if all required stages are present and strictly increasing
monotonic_mask = (
    pieces["lifetime_2nd_strike_s"].notna()
    & pieces["lifetime_3rd_strike_s"].notna()
    & pieces["lifetime_4th_strike_s"].notna()
    & pieces["lifetime_auxiliary_press_s"].notna()
    & pieces["lifetime_bath_s"].notna()
    & (pieces["lifetime_2nd_strike_s"] < pieces["lifetime_3rd_strike_s"])
    & (pieces["lifetime_3rd_strike_s"] < pieces["lifetime_4th_strike_s"])
    & (pieces["lifetime_4th_strike_s"] < pieces["lifetime_auxiliary_press_s"])
    & (pieces["lifetime_auxiliary_press_s"] < pieces["lifetime_bath_s"])
)

cleaning_report["rows_removed_non_monotonic"] = int((~monotonic_mask).sum())

non_monotonic_examples = pieces.loc[
    ~monotonic_mask,
    ["timestamp", "piece_id", "die_matrix"] + monotonic_cols
].head(20)

pieces = pieces.loc[monotonic_mask].copy()

print("Rows removed due to non-monotonic cumulative order:", cleaning_report["rows_removed_non_monotonic"])
print("Remaining pieces after monotonic validation:", len(pieces))

display(non_monotonic_examples)
display(pieces.head(10))


Rows removed due to non-monotonic cumulative order: 28750
Remaining pieces after monotonic validation: 140404


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s
74,2025-11-06 15:42:45.207000+00:00,251106001799,5052,18.100000,NaN,38.000000,55.000000,56.599998
366,2025-11-06 17:29:34.925000+00:00,251106000000,5052,NaN,NaN,39.299999,56.400002,58.000000
447,2025-11-06 17:48:10.024000+00:00,251106000000,5052,NaN,NaN,37.000000,54.200001,55.799999
522,2025-11-06 18:10:25.071000+00:00,251106000000,5052,NaN,29.600000,42.900002,59.599998,61.299999
631,2025-11-06 18:35:51.233000+00:00,251106000000,5052,NaN,24.799999,38.099998,55.099998,56.700001
1246,2025-11-06 23:35:41.608000+00:00,251106000000,5052,NaN,NaN,38.900002,56.000000,57.599998
1326,2025-11-06 23:56:58.758000+00:00,251106000000,5052,NaN,NaN,39.000000,55.799999,57.400002
1501,2025-11-07 00:42:40.870000+00:00,251107000000,5052,NaN,24.500000,38.000000,55.299999,56.900002
1740,2025-11-07 01:40:45.196000+00:00,251107003854,5052,18.500000,NaN,NaN,NaN,NaN
1757,2025-11-07 01:50:42.181000+00:00,251107000000,5052,NaN,NaN,38.900002,56.200001,57.900002


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s
1,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001
2,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002
3,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002
4,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998
5,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001
6,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,23.400000,36.799999,53.599998,55.200001,55.200001
7,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,24.900000,38.200001,54.700001,56.400002,56.400002
8,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,24.500000,37.700001,54.700001,56.299999,56.299999
12,2025-11-06 15:27:57.427000+00:00,251106001733,5052,16.700001,23.299999,36.599998,53.299999,54.900002,54.900002
16,2025-11-06 15:29:04.470000+00:00,251106001738,5052,16.700001,23.600000,36.799999,54.099998,55.799999,55.799999


## Step 8: Compute OEE cycle time and filter

The OEE cycle time measures the **time between consecutive pieces** exiting the line — it is a production throughput metric. The original PLC computes it as the rolling average of the last 5 pieces at the auxiliary press.

Since the auxiliary press signal is not in our dataset, we approximate it from the **timestamp intervals** between consecutive pieces, using a rolling window of 5.

Valid OEE cycle time is **11–16 seconds**. Values outside this range indicate production stops, changeovers, or sensor errors. Pieces with invalid OEE are kept in silver (they are valid pieces) but their `oee_cycle_time_s` is set to NULL.

In [10]:
pieces = pieces.sort_values("timestamp").reset_index(drop=True).copy()

# Inter-piece interval in seconds
pieces["inter_piece_interval_s"] = pieces["timestamp"].diff().dt.total_seconds()

# Rolling mean of the last 5 inter-piece intervals
pieces["oee_cycle_time_s"] = (
    pieces["inter_piece_interval_s"]
    .rolling(window=5, min_periods=5)
    .mean()
)

# Mark production gaps (> 5 minutes) for downstream analysis
pieces["production_gap_after_prev"] = pieces["inter_piece_interval_s"] > 300

# Keep the piece, but nullify invalid OEE values
invalid_oee_mask = (
    pieces["oee_cycle_time_s"].notna()
    & (
        (pieces["oee_cycle_time_s"] < 11)
        | (pieces["oee_cycle_time_s"] > 16)
    )
)

cleaning_report["oee_invalid_set_to_null"] = int(invalid_oee_mask.sum())
cleaning_report["production_gaps_gt_5min"] = int(pieces["production_gap_after_prev"].sum())

pieces.loc[invalid_oee_mask, "oee_cycle_time_s"] = np.nan

print("Pieces with invalid OEE set to NULL:", cleaning_report["oee_invalid_set_to_null"])
print("Production gaps > 5 min:", cleaning_report["production_gaps_gt_5min"])

display(
    pieces[
        [
            "timestamp",
            "piece_id",
            "die_matrix",
            "inter_piece_interval_s",
            "oee_cycle_time_s",
            "production_gap_after_prev",
        ]
    ].head(15)
)


Pieces with invalid OEE set to NULL: 34848
Production gaps > 5 min: 775


,timestamp,piece_id,die_matrix,inter_piece_interval_s,oee_cycle_time_s,production_gap_after_prev
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,NaN,NaN,False
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,12.708,NaN,False
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,14.609,NaN,False
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,12.719,NaN,False
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,14.000,NaN,False
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,13.309,13.4690,False
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,12.611,13.4496,False
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,13.713,13.2704,False
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,67.332,NaN,False
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,67.043,NaN,False


## Step 9: Write to silver

Append the cleaned, validated pieces to `silver.clean_pieces`.

In [17]:
# -----------------------------
# Final column selection (silver schema)
# -----------------------------
final_cols = [
    "timestamp",
    "piece_id",
    "die_matrix",
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
    "lifetime_general_s",
    "oee_cycle_time_s",
]

final_cols = [c for c in final_cols if c in pieces.columns]
silver_df = pieces[final_cols].copy()

print("Final dataset shape (to be written to silver):", silver_df.shape)
display(silver_df.head(5))

# Write directly to silver.clean_pieces
with engine.begin() as conn:
    # Optional: clear table first for a clean rebuild
    conn.execute(text("TRUNCATE TABLE silver.clean_pieces"))
    
    silver_df.to_sql(
        "clean_pieces",
        conn,
        schema="silver",
        if_exists="append",
        index=False,
        method="multi",
        chunksize=5000,
    )

print("Write complete.")

Final dataset shape (to be written to silver): (140404, 10)


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,oee_cycle_time_s
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,NaN
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,NaN
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,NaN
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,NaN
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,NaN


Write complete.


In [19]:
pd.read_sql("""
SELECT COUNT(*) AS n
FROM silver.clean_pieces
""", engine)
pd.read_sql("""
SELECT *
FROM silver.clean_pieces
LIMIT 5
""", engine)

,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_bath_s,lifetime_general_s,processed_at,oee_cycle_time_s,lifetime_auxiliary_press_s
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,56.200001,56.200001,2026-04-14 02:34:28.839803+00:00,None,54.599998
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,56.400002,56.400002,2026-04-14 02:34:28.839803+00:00,None,54.799999
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,56.900002,56.900002,2026-04-14 02:34:28.839803+00:00,None,55.299999
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,57.099998,57.099998,2026-04-14 02:34:28.839803+00:00,None,55.500000
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,57.200001,57.200001,2026-04-14 02:34:28.839803+00:00,None,55.500000


## Cleaning Report

Summary of all cleaning decisions applied in this run, with counts and justifications.

# 🧹 Cleaning Report — Bronze → Silver

This section summarizes how raw PLC signals were cleaned and transformed into a reliable dataset of valid pieces (`silver.clean_pieces`).  

The goal is simple: keep only physically meaningful, traceable, and comparable pieces — and remove everything that comes from sensor errors, tracking failures, or machine interruptions.

---

## 1. Starting point: raw PLC signals

We begin with two raw tables:

- `bronze.raw_lifetime` — cumulative travel times (signal/value format)
- `bronze.raw_piece_info` — piece identification (piece_id, die_matrix)

These tables reflect exactly what the PLC recorded — including noise, errors, and inconsistencies.

---

## 2. Removing the upsetting signal (1st strike)

The first step was to completely discard the **upsetting (1st strike)** signal.

Why?

- Values are unrealistically low (~0–6s vs ~18s for the next stage)
- ~22.8% of values are zero
- The distribution shows no meaningful physical behavior

👉 Conclusion: this signal is **incorrectly computed at the PLC level** and cannot be trusted.

---

## 3. Removing zero values

All records with value = 0 were removed.

These do **not** represent fast pieces — they represent:

- missed sensor detections  
- tracking failures  
- incomplete PLC cycles  

Keeping them would distort distributions and bias minimum values.

---

## 4. Deduplicating timestamps

The PLC occasionally records the same signal twice at the same timestamp.

We kept only the **last reading per signal per timestamp**.

This ensures:

- no double-counting of pieces  
- correct production frequency  
- consistent downstream aggregation  

---

## 5. Pivoting and joining into pieces

The raw data is stored in a *tall format* (`timestamp / signal / value`).

We transformed it into a **wide format**:

- one row per timestamp (= one piece)
- one column per process stage
- joined with:
  - `piece_id`
  - `die_matrix`

This creates the foundation of the silver dataset: **one row = one physical piece**.

---

## 6. Dropping pieces without identification

Rows missing either:

- `piece_id`
- `die_matrix`

were removed.

Why?

- Without `piece_id` → no traceability  
- Without `die_matrix` → cannot segment analysis  

👉 And segmentation by die matrix is essential, because each matrix has different timing behavior.

---

## 7. Removing extreme outliers (per die matrix)

For each die matrix and each signal, we computed:

- Q1, Q3, and IQR  
- threshold = Q3 + 3×IQR  

Any piece outside this range was removed.

These extreme values (often 10–27× normal) are not slow production — they represent:

- stuck pieces  
- unclosed tracking cycles  
- machine stops during processing  

👉 Removing them ensures we model **normal operating behavior**, not failures.

---

## 8. Validating physical process order

Each piece must follow the physical sequence:
# 🧹 Cleaning Report — Bronze → Silver

This section summarizes how raw PLC signals were cleaned and transformed into a reliable dataset of valid pieces (`silver.clean_pieces`).  

The goal is simple: keep only physically meaningful, traceable, and comparable pieces — and remove everything that comes from sensor errors, tracking failures, or machine interruptions.

---

## 1. Starting point: raw PLC signals

We begin with two raw tables:

- `bronze.raw_lifetime` — cumulative travel times (signal/value format)
- `bronze.raw_piece_info` — piece identification (piece_id, die_matrix)

These tables reflect exactly what the PLC recorded — including noise, errors, and inconsistencies.

---

## 2. Removing the upsetting signal (1st strike)

The first step was to completely discard the **upsetting (1st strike)** signal.

Why?

- Values are unrealistically low (~0–6s vs ~18s for the next stage)
- ~22.8% of values are zero
- The distribution shows no meaningful physical behavior

👉 Conclusion: this signal is **incorrectly computed at the PLC level** and cannot be trusted.

---

## 3. Removing zero values

All records with value = 0 were removed.

These do **not** represent fast pieces — they represent:

- missed sensor detections  
- tracking failures  
- incomplete PLC cycles  

Keeping them would distort distributions and bias minimum values.

---

## 4. Deduplicating timestamps

The PLC occasionally records the same signal twice at the same timestamp.

We kept only the **last reading per signal per timestamp**.

This ensures:

- no double-counting of pieces  
- correct production frequency  
- consistent downstream aggregation  

---

## 5. Pivoting and joining into pieces

The raw data is stored in a *tall format* (`timestamp / signal / value`).

We transformed it into a **wide format**:

- one row per timestamp (= one piece)
- one column per process stage
- joined with:
  - `piece_id`
  - `die_matrix`

This creates the foundation of the silver dataset: **one row = one physical piece**.

---

## 6. Dropping pieces without identification

Rows missing either:

- `piece_id`
- `die_matrix`

were removed.

Why?

- Without `piece_id` → no traceability  
- Without `die_matrix` → cannot segment analysis  

👉 And segmentation by die matrix is essential, because each matrix has different timing behavior.

---

## 7. Removing extreme outliers (per die matrix)

For each die matrix and each signal, we computed:

- Q1, Q3, and IQR  
- threshold = Q3 + 3×IQR  

Any piece outside this range was removed.

These extreme values (often 10–27× normal) are not slow production — they represent:

- stuck pieces  
- unclosed tracking cycles  
- machine stops during processing  

👉 Removing them ensures we model **normal operating behavior**, not failures.

---

## 8. Validating physical process order

Each piece must follow the physical sequence:
2nd strike < 3rd strike < 4th strike < auxiliary press < bath

If this order is violated, the piece is removed.

Why?

- All values are cumulative time from furnace exit  
- The process is strictly sequential  
- A violation is physically impossible  

👉 These cases indicate **sensor misattribution or PLC timing errors**.

---

## 9. Computing OEE cycle time

We computed the OEE cycle time as:

- time difference between consecutive pieces  
- rolling average over the last 5 pieces  

Expected range:

- **11–16 seconds per piece**

Values outside this range were set to **NULL** (but the piece is kept).

Why?

- These values usually correspond to:
  - production stops  
  - maintenance  
  - die matrix changeovers  

👉 The piece is valid — but the throughput metric is not.

---

## 10. Final result

After applying all cleaning steps:

- The dataset contains only **valid, physically consistent pieces**
- Each row represents a real piece with:
  - full process timing
  - correct identification
  - no sensor noise or PLC artifacts

Expected size:

- ~160,000–170,000 pieces (from ~180,000 raw)

---

My takeaway

The cleaning process reflects **how the production line actually works**.

We are not removing “bad data” arbitrarily.  
We are removing:

- things that **never physically happened**
- things that the **PLC failed to measure correctly**
- things that correspond to **abnormal machine states**

👉 What remains is a dataset that truly represents **normal forging operations**, ready for analysis and modeling.
